In [1]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [6]:
df = pd.read_csv("zomato.csv", encoding="latin1")
print(df.head())

   Restaurant ID         Restaurant Name  Country Code              City  \
0        6317637        Le Petit Souffle           162       Makati City   
1        6304287        Izakaya Kikufuji           162       Makati City   
2        6300002  Heat - Edsa Shangri-La           162  Mandaluyong City   
3        6318506                    Ooma           162  Mandaluyong City   
4        6314302             Sambo Kojin           162  Mandaluyong City   

                                             Address  \
0  Third Floor, Century City Mall, Kalayaan Avenu...   
1  Little Tokyo, 2277 Chino Roces Avenue, Legaspi...   
2  Edsa Shangri-La, 1 Garden Way, Ortigas, Mandal...   
3  Third Floor, Mega Fashion Hall, SM Megamall, O...   
4  Third Floor, Mega Atrium, SM Megamall, Ortigas...   

                                     Locality  \
0   Century City Mall, Poblacion, Makati City   
1  Little Tokyo, Legaspi Village, Makati City   
2  Edsa Shangri-La, Ortigas, Mandaluyong City   
3      SM 

In [7]:
df = df[['Rating text', 'Has Online delivery']]
print(df.head())

  Rating text Has Online delivery
0   Excellent                  No
1   Excellent                  No
2   Very Good                  No
3   Excellent                  No
4   Excellent                  No


In [8]:
print(df.isnull().sum())
df.dropna(inplace=True)

Rating text            0
Has Online delivery    0
dtype: int64


In [9]:
df['Has Online delivery'] = df['Has Online delivery'].map({
    'Yes':1,
    'No':0
})
print(df.head())

  Rating text  Has Online delivery
0   Excellent                    0
1   Excellent                    0
2   Very Good                    0
3   Excellent                    0
4   Excellent                    0


In [10]:
ps = PorterStemmer()
corpus = []
for text in df['Rating text']:
    text = re.sub('[^a-zA-Z]', ' ', str(text))
    text = text.lower()
    text = text.split()
    text = [ps.stem(word) for word in text
            if word not in stopwords.words('english')]
    text = ' '.join(text)
    corpus.append(text)

In [11]:
cv = CountVectorizer()
X = cv.fit_transform(corpus).toarray()
y = df['Has Online delivery']

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [14]:
model = MultinomialNB()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)


In [15]:
print("Accuracy :", accuracy_score(y_test, y_pred))

Accuracy : 0.7430664573521716


In [16]:
cm = confusion_matrix(y_test, y_pred)
print(cm)

[[1403   13]
 [ 478   17]]


In [17]:
review = "Excellent"
review = re.sub('[^a-zA-Z]', ' ', review)
review = review.lower()
review = review.split()
review = [ps.stem(word) for word in review
          if word not in stopwords.words('english')]
review = ' '.join(review)
review = cv.transform([review]).toarray()
prediction = model.predict(review)
if prediction[0] == 1:
    print("Restaurant has Online Delivery")
else:
    print("Restaurant has No Online Delivery")

Restaurant has No Online Delivery
